# 🔍 Data Validation & Recovery Notebook

This notebook performs comprehensive data validation and **provides recovery capabilities**.

## Modes
- **Validation Mode**: Identify all issues across tables
- **Recovery Mode**: Fix issues by re-running inference/evaluation

## Tables Checked
| Table | Purpose |
|-------|--------|
| `vlm_samples` | Input samples (prompts, ground truth) |
| `vlm_images` | Image binary data |
| `vlm_responses` | Model outputs (per sample × model) |
| `vlm_evaluations` | Glider & VLM Judge scores |

In [ ]:
# === Cell 1: Setup & Imports ===
import sys
from pathlib import Path

# Path setup (notebook is at artemis_final/notebooks/ares/)
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/

for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import os
import time
import json
import hashlib
import logging
import threading
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Set, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sqlalchemy import text
from IPython.display import display, HTML, Markdown

# Suppress HTTP noise
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-15s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('VALIDATION')

print(f"📁 NOTEBOOK_DIR: {NOTEBOOK_DIR}")
print(f"📁 ARTEMIS_DIR: {ARTEMIS_DIR}")
print("✅ Imports done!")

In [ ]:
# === Cell 2: Configuration ===
@dataclass
class ValidationConfig:
    """Validation and recovery settings."""
    # Validation settings
    expected_models: int = 5  # Number of models per sample
    
    # Recovery settings
    dry_run: bool = True  # If True, only report issues without fixing
    max_recovery_samples: int = 100  # Max samples to recover per run
    
    # Inference settings (for response recovery)
    models_yaml: str = str(ARTEMIS_DIR / 'ares' / 'configs' / 'models.yaml')
    temperature: float = 0.0
    max_tokens: int = 512
    max_workers: int = 8
    batch_size: int = 25
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    run_id: str = field(default_factory=lambda: f"recovery_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    
    # GPU endpoints for metrics (optional)
    gpu_endpoints: Dict[str, str] = field(default_factory=lambda: {
        'deepseek_ocr': 'http://localhost:9002/metrics',
        'qwen2_5_vl_3b': 'http://localhost:9001/metrics',
        'qwen2_5_vl_7b': 'http://localhost:9001/metrics',
        'qwen3_vl_8b_thinking': 'http://localhost:9000/metrics',
        'gemma_3_27b': 'http://localhost:9000/metrics',
    })
    
    # Endpoint settings (for evaluation recovery)
    glider_ports: List[int] = field(default_factory=lambda: [8806])
    vlm_judge_ports: List[int] = field(default_factory=lambda: [8807])

config = ValidationConfig()

print("⚙️ Configuration:")
print(f"   🔒 Dry Run: {config.dry_run} {'(will NOT modify data)' if config.dry_run else '⚠️ WILL MODIFY DATA'}")
print(f"   📊 Max recovery samples: {config.max_recovery_samples}")
print(f"   🤖 Expected models per sample: {config.expected_models}")

In [ ]:
# === Cell 3: Database Connection ===
from ares.db.connection import get_engine, test_connection
from ares.configs.db_config import MODEL_NAMES

print("🔗 Testing database connection...")
if not test_connection():
    raise RuntimeError("Database connection failed!")

engine = get_engine()
logger.info("Database connected!")
logger.info(f"Models: {MODEL_NAMES}")

---
# 📊 Section 1: Validation Checks

Run all validation checks to identify issues.

In [ ]:
# === Cell 4: Table Overview ===
tables = ['vlm_samples', 'vlm_images', 'vlm_responses', 'vlm_evaluations']
row_counts = {}

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        row_counts[table] = result.fetchone()[0]

print("📊 Table Row Counts:")
print("=" * 50)
for table, count in row_counts.items():
    print(f"{table:25} {count:>15,} rows")
print("=" * 50)

# Get model counts
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT model_name, COUNT(*) as cnt 
        FROM vlm_responses 
        GROUP BY model_name 
        ORDER BY cnt DESC
    """))
    models_df = pd.DataFrame(result.fetchall(), columns=['model_name', 'count'])

print("\n🤖 Models:")
display(models_df)

In [ ]:
# === Cell 5: Issue 1 - Missing Responses ===
# Find samples that don't have all 5 model responses

query_incomplete_samples = """
WITH response_counts AS (
    SELECT 
        s.sample_id,
        s.source_config,
        s.data_split,
        COUNT(DISTINCT r.model_name) as model_count,
        ARRAY_AGG(DISTINCT r.model_name) as models_present
    FROM vlm_samples s
    LEFT JOIN vlm_responses r ON s.sample_id = r.sample_id AND r.ok = true
    GROUP BY s.sample_id, s.source_config, s.data_split
)
SELECT * FROM response_counts
WHERE model_count < :expected_models
ORDER BY source_config, model_count
"""

with engine.connect() as conn:
    incomplete_df = pd.read_sql(
        text(query_incomplete_samples), 
        conn, 
        params={'expected_models': config.expected_models}
    )

print(f"❌ Issue 1 - Missing Responses: {len(incomplete_df):,} samples with < {config.expected_models} models")
print("=" * 70)

if len(incomplete_df) > 0:
    # By source config
    by_config = incomplete_df.groupby('source_config').size().reset_index(name='count')
    print("\n📊 By Source Config:")
    display(by_config.head(20))
    
    # By model count
    by_model_count = incomplete_df.groupby('model_count').size().reset_index(name='count')
    print("\n📊 By Model Count:")
    display(by_model_count)
    
    print(f"\n📝 Sample (first 10):")
    display(incomplete_df.head(10))
else:
    print("✅ All samples have responses from all models!")

In [ ]:
# === Cell 6: Issue 2 - Missing Evaluations ===
# Find responses that don't have corresponding evaluations

query_missing_evals = """
SELECT 
    r.sample_id,
    r.model_name,
    s.source_config,
    s.data_split
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true AND e.sample_id IS NULL
ORDER BY s.source_config
"""

with engine.connect() as conn:
    missing_evals_df = pd.read_sql(text(query_missing_evals), conn)

print(f"❌ Issue 2 - Missing Evaluations: {len(missing_evals_df):,} responses without evaluations")
print("=" * 70)

if len(missing_evals_df) > 0:
    by_config = missing_evals_df.groupby('source_config').size().reset_index(name='count')
    print("\n📊 By Source Config:")
    display(by_config.head(20))
    
    by_model = missing_evals_df.groupby('model_name').size().reset_index(name='count')
    print("\n📊 By Model:")
    display(by_model)
else:
    print("✅ All responses have evaluations!")

In [ ]:
# === Cell 7: Issue 3 - Missing Glider/VLM Judge Scores ===

query_missing_scores = """
SELECT 
    COUNT(*) as total_evals,
    SUM(CASE WHEN glider_score IS NULL THEN 1 ELSE 0 END) as missing_glider,
    SUM(CASE WHEN judge_molmo_score IS NULL THEN 1 ELSE 0 END) as missing_vlm_judge,
    SUM(CASE WHEN glider_score IS NOT NULL AND judge_molmo_score IS NOT NULL THEN 1 ELSE 0 END) as complete
FROM vlm_evaluations
"""

with engine.connect() as conn:
    score_stats = pd.read_sql(text(query_missing_scores), conn)

print("📊 Evaluation Score Coverage:")
print("=" * 50)
display(score_stats.T)

missing_glider = int(score_stats['missing_glider'].iloc[0])
missing_vlm_judge = int(score_stats['missing_vlm_judge'].iloc[0])

if missing_glider > 0:
    print(f"\n⚠️ {missing_glider:,} evaluations missing Glider score")
if missing_vlm_judge > 0:
    print(f"⚠️ {missing_vlm_judge:,} evaluations missing VLM Judge score")

In [ ]:
# === Cell 8: Validation Summary ===

print("\n" + "=" * 70)
print("📋 VALIDATION SUMMARY")
print("=" * 70)
print(f"\n1️⃣  Missing Responses:    {len(incomplete_df):>8,} samples")
print(f"2️⃣  Missing Evaluations:  {len(missing_evals_df):>8,} responses")
print(f"3️⃣  Missing Glider:       {missing_glider:>8,} evals")
print(f"4️⃣  Missing VLM Judge:    {missing_vlm_judge:>8,} evals")
print("\n" + "=" * 70)

total_issues = len(incomplete_df) + len(missing_evals_df)
if total_issues == 0:
    print("\n✅ All data is complete! No recovery needed.")
else:
    print(f"\n⚠️ Total issues requiring recovery: {total_issues:,}")
    if config.dry_run:
        print("\n💡 Set `config.dry_run = False` to enable recovery.")

---
# 🔧 Section 2: Response Recovery

Re-run VLM inference for samples with missing responses.

In [ ]:
# === Cell 9: Initialize VLM Clients for Response Recovery ===
config.dry_run = False 
if config.dry_run:
    print("⏭️ Skipping VLM client initialization (dry run mode)")
    vlm_client = None
    gpu_client = None
    scorer = None
    model_specs = None
else:
    print("🔧 Initializing VLM clients for response recovery...")
    
    from inference_engine.client import WhichVLMClient
    from ares.evaluation.evaluation import Scorer
    from ares.metrics.metrics_client import GPUMetricsClient
    from ares.utils.common_utils import return_model_specs
    
    try:
        vlm_client = WhichVLMClient.from_yaml(config.models_yaml)
        gpu_client = GPUMetricsClient(endpoints=config.gpu_endpoints)
        scorer = Scorer()
        model_specs = return_model_specs()
        print(f"✅ VLM clients initialized!")
        print(f"   Models: {MODEL_NAMES}")
    except Exception as e:
        logger.error(f"Failed to initialize VLM clients: {e}")
        vlm_client = None

In [ ]:
# === Cell 10: Response Recovery Execution ===

from ares.db.operations import insert_responses, insert_samples, insert_images, get_existing_responses
from ares.evaluation.sample_processor import process_sample_normalized
from ares.data.dataset_loader import CauldronLoader

recovery_logger = logging.getLogger('RECOVERY')

if config.dry_run:
    print("⏭️ Skipping response recovery (dry run)")
    print(f"   Would attempt to recover up to {min(len(incomplete_df), config.max_recovery_samples)} samples")
elif len(incomplete_df) == 0:
    print("✅ No missing responses to recover!")
elif vlm_client is None:
    print("❌ Cannot recover responses: VLM clients not initialized")
else:
    print(f"🔧 Recovering responses for up to {config.max_recovery_samples} samples...")
    print("=" * 70)
    
    # Group by source_config for efficient processing
    samples_to_recover = incomplete_df.head(config.max_recovery_samples)
    by_config = samples_to_recover.groupby('source_config')
    
    total_recovered = 0
    total_errors = 0
    db_lock = threading.Lock()
    
    for source_config, group in tqdm(by_config, desc="Source configs"):
        recovery_logger.info(f"[{source_config}] Processing {len(group)} samples")
        
        # Load samples from this config
        try:
            samples = CauldronLoader.load_samples(
                source_config, 
                n_samples=len(group) + 50,  # Load extra to find matches
                random_sample=False
            )
        except Exception as e:
            recovery_logger.error(f"[{source_config}] Failed to load samples: {e}")
            total_errors += len(group)
            continue
        
        sample_batch, image_batch, response_batch = [], [], []
        
        for _, row in group.iterrows():
            sample_id = row['sample_id']
            models_present = row['models_present'] if row['models_present'] else []
            models_missing = [m for m in MODEL_NAMES if m not in models_present]
            
            if not models_missing:
                continue
            
            # Find the sample by index from sample_id
            try:
                idx = int(sample_id.split('_')[1]) if '_' in sample_id else 0
                if idx >= len(samples):
                    recovery_logger.warning(f"[{source_config}] Sample index {idx} out of range")
                    total_errors += 1
                    continue
                    
                sample = samples[idx]
                
                recovery_logger.info(f"[{source_config}] {sample_id}: recovering models {models_missing}")
                
                result = process_sample_normalized(
                    sample=sample,
                    sample_idx=idx,
                    source_config=source_config,
                    vlm_client=vlm_client,
                    gpu_client=gpu_client,
                    scorer=scorer,
                    config=config,
                    model_specs=model_specs,
                    models_to_run=models_missing,
                )
                
                if result:
                    sample_record, image_record, response_records = result
                    # Only insert new responses, not sample/image (they may already exist)
                    response_batch.extend(response_records)
                    total_recovered += len(response_records)
                    recovery_logger.info(f"[{source_config}] ✓ Recovered {len(response_records)} responses")
                else:
                    total_errors += 1
                    
            except Exception as e:
                recovery_logger.error(f"[{source_config}] Error processing {sample_id}: {e}")
                total_errors += 1
                continue
            
            # Batch insert
            if len(response_batch) >= config.batch_size:
                with db_lock:
                    insert_responses(response_batch)
                response_batch = []
        
        # Final batch for this config
        if response_batch:
            with db_lock:
                insert_responses(response_batch)
            response_batch = []
    
    print("\n" + "=" * 70)
    print(f"📊 Response Recovery Complete:")
    print(f"   ✅ Recovered: {total_recovered} responses")
    print(f"   ❌ Errors: {total_errors}")

---
# 🔧 Section 3: Evaluation Recovery

Re-run Glider and VLM Judge for responses missing evaluations.

In [ ]:
# === Cell 11: Initialize Evaluation Pipeline ===

if config.dry_run:
    print("⏭️ Skipping evaluation pipeline initialization (dry run mode)")
    eval_pipeline = None
elif len(missing_evals_df) == 0:
    print("✅ No missing evaluations to recover!")
    eval_pipeline = None
else:
    print("🔧 Initializing evaluation pipeline...")
    
    from inference_engine.runners import OpenAIStyleRunner
    from inference_engine.config import ModelEndpoint
    from ares.evaluation.router_eval_pipeline import RouterEvalPipeline
    
    try:
        # Configure endpoints
        endpoints = []
        for i, port in enumerate(config.glider_ports):
            endpoints.append(ModelEndpoint(
                name=f"glider-{i+1}",
                model_id="PatronusAI/glider",
                base_url=f"http://localhost:{port}/v1",
                api_key="EMPTY",
                pricing={},
                extra_params={}
            ))
        
        for i, port in enumerate(config.vlm_judge_ports):
            endpoints.append(ModelEndpoint(
                name=f"vlm-judge-{i+1}",
                model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
                base_url=f"http://localhost:{port}/v1",
                api_key="EMPTY",
                pricing={},
                extra_params={}
            ))
        
        runner = OpenAIStyleRunner(
            models=endpoints,
            request_timeout_s=180,
            max_workers=64
        )
        
        eval_pipeline = RouterEvalPipeline(
            engine=engine,
            runner=runner,
            glider_model_names=[f"glider-{i+1}" for i in range(len(config.glider_ports))],
            vlm_judge_model_names=[f"vlm-judge-{i+1}" for i in range(len(config.vlm_judge_ports))],
            tracker_path="recovery_eval_progress.json",
            use_glider=True,
            use_vlm_judge=True,
        )
        
        print(f"✅ Evaluation pipeline initialized!")
        print(f"   Glider endpoints: {config.glider_ports}")
        print(f"   VLM Judge endpoints: {config.vlm_judge_ports}")
        
    except Exception as e:
        logger.error(f"Failed to initialize evaluation pipeline: {e}")
        eval_pipeline = None

In [ ]:
# === Cell 12: Evaluation Recovery Execution ===

if config.dry_run:
    print("⏭️ Skipping evaluation recovery (dry run)")
    print(f"   Would attempt to recover up to {min(len(missing_evals_df), config.max_recovery_samples * 5)} evaluations")
elif eval_pipeline is None:
    print("⏭️ Cannot run evaluation recovery: pipeline not initialized")
else:
    print(f"🔧 Recovering evaluations...")
    print("=" * 70)
    
    # Get unique source configs from missing evaluations
    configs_to_process = missing_evals_df['source_config'].unique().tolist()
    
    # Limit processing
    max_configs = min(len(configs_to_process), 10)  # Process up to 10 configs at a time
    configs_to_process = configs_to_process[:max_configs]
    
    print(f"   Processing {len(configs_to_process)} source configs")
    
    for source_config in tqdm(configs_to_process, desc="Evaluating configs"):
        try:
            eval_pipeline.process_source_config(
                source_config=source_config,
                batch_size=50,
                force=False,  # Only process missing evaluations
                pbar=None
            )
            recovery_logger.info(f"[{source_config}] ✓ Evaluation complete")
        except Exception as e:
            recovery_logger.error(f"[{source_config}] Evaluation failed: {e}")
    
    print("\n✅ Evaluation recovery complete!")

---
# 📋 Section 4: Final Verification

Re-run validation to confirm fixes.

In [ ]:
# === Cell 13: Post-Recovery Validation ===

if not config.dry_run:
    print("🔄 Re-running validation checks...\n")
    
    # Check responses
    with engine.connect() as conn:
        incomplete_after = pd.read_sql(
            text(query_incomplete_samples), 
            conn, 
            params={'expected_models': config.expected_models}
        )
    
    # Check evaluations
    with engine.connect() as conn:
        missing_evals_after = pd.read_sql(text(query_missing_evals), conn)
    
    print("\n" + "=" * 70)
    print("📊 BEFORE vs AFTER Recovery")
    print("=" * 70)
    print(f"\nMissing Responses:")
    print(f"   Before: {len(incomplete_df):>8,}")
    print(f"   After:  {len(incomplete_after):>8,}")
    print(f"   Fixed:  {len(incomplete_df) - len(incomplete_after):>8,}")
    
    print(f"\nMissing Evaluations:")
    print(f"   Before: {len(missing_evals_df):>8,}")
    print(f"   After:  {len(missing_evals_after):>8,}")
    print(f"   Fixed:  {len(missing_evals_df) - len(missing_evals_after):>8,}")
    print("=" * 70)
else:
    print("⏭️ Skipping post-recovery validation (dry run mode)")
    print("\n💡 To run actual recovery:")
    print("   1. Set config.dry_run = False in Cell 2")
    print("   2. Re-run all cells")

In [ ]:
# === Cell 14: Final Summary ===

print("\n" + "=" * 70)
print("🏁 DATA VALIDATION & RECOVERY COMPLETE")
print("=" * 70)
print(f"\nRun ID: {config.run_id}")
print(f"Dry Run: {config.dry_run}")
print(f"\nFinal Table Counts:")

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.fetchone()[0]
        print(f"   {table:25} {count:>12,} rows")

print("=" * 70)